# SmartCart: Part 2: User Based Collaborative Filtering

##  Imports

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
# pandas             : data loading and DataFrame manipulation
# numpy              : numerical operations and random seed for reproducibility
# cosine_similarity  : computes similarity between user rating vectors
# matplotlib/seaborn : visualisation
# defaultdict        : cleaner accumulation of weighted scores (no KeyError on first access)
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import defaultdict

pd.set_option('display.max_columns', None)  # show all columns when printing DataFrames
np.random.seed(42)                          # fix random seed for reproducible splits


## 2.1  Load Data & Build User Item Matrix

In [ ]:
# ── Load & Prepare Data ───────────────────────────────────────────────────
# Re-load from source so Part 2 can run independently from Part 1
user_data    = pd.read_csv('ecommerce_user_data.csv')
product_data = pd.read_csv('product_details.csv')

# Parse timestamps so they are proper datetime objects (not plain strings)
user_data['Timestamp'] = pd.to_datetime(user_data['Timestamp'])

# Remove duplicate (UserID, ProductID) pairs — keep the most recent rating
# A user may have interacted with the same product more than once;
# keeping the last ensures the matrix reflects their current preference.
user_data = user_data.drop_duplicates(subset=['UserID', 'ProductID'], keep='last')

# ── Build User–Item Matrix ────────────────────────────────────────────────
# Rows = users, Columns = products, Values = rating (1–5)
# Cells with no interaction are NaN initially
user_item_matrix = user_data.pivot_table(
    index='UserID', columns='ProductID', values='Rating'
)

# Replace NaN with 0 — cosine similarity needs numeric values.
# 0 means 'no interaction', NOT a rating of zero.
user_item_matrix_filled = user_item_matrix.fillna(0)

# ── Quick sanity check ───────────────────────────────────────────────────
n_users, n_products = user_item_matrix_filled.shape
n_rated   = (user_item_matrix_filled != 0).sum().sum()   # total non-zero entries
sparsity  = 1 - n_rated / (n_users * n_products)         # fraction of missing ratings

print(f"Users    : {n_users}")
print(f"Products : {n_products}")
print(f"Ratings  : {n_rated}")
print(f"Sparsity : {sparsity*100:.1f}%  (fraction of matrix that is 0)")
user_item_matrix_filled.head()


## 2.2  Cosine Similarity Between Users

For two users **u** and **v** with rating vectors **r_u** and **r_v**:

$$\text{sim}(u, v) = \frac{\mathbf{r}_u \cdot \mathbf{r}_v}{\|\mathbf{r}_u\| \cdot \|\mathbf{r}_v\|}$$

Values range from 0 (orthogonal / no overlap) to 1 (identical taste).

In [ ]:
# ── Compute User–User Cosine Similarity ──────────────────────────────────
# Each user is represented as a 100-dimensional vector (one value per product).
# Cosine similarity measures the angle between two vectors:
#   sim = 1  → identical rating patterns
#   sim = 0  → no products in common (orthogonal vectors)
# Result is a 50×50 matrix where entry [i,j] = similarity between user i and user j.
sim_matrix = cosine_similarity(user_item_matrix_filled)

# Wrap in a DataFrame with UserID labels so we can look up by name
similarity_df = pd.DataFrame(
    sim_matrix,
    index=user_item_matrix_filled.index,
    columns=user_item_matrix_filled.index
)

print("Similarity matrix shape:", similarity_df.shape)   # should be (50, 50)

# Sanity check: drop U000 from its own row (self-similarity = 1, not useful)
# then show the 5 users most similar to U000
print("\nTop-5 most similar users to U000:")
print(similarity_df['U000'].drop('U000').sort_values(ascending=False).head())


## 2.3  Recommendation Function

**Algorithm:**
1. Find the `top_n_users` most similar users to the target user.
2. For each product the target user has **not** rated, compute a weighted predicted score:
$$\hat{r}_{u,p} = \frac{\sum_{v \in N(u)} \text{sim}(u,v) \cdot r_{v,p}}{\sum_{v \in N(u)} \text{sim}(u,v)}$$
3. Return the top-N products by predicted score.

In [ ]:
# ── Recommendation Function (Weighted k-NN) ───────────────────────────────
# Steps:
#  1. Find the k most similar users to the target (nearest neighbours).
#  2. For each product the target has NOT rated, compute a predicted score:
#       predicted_score(p) = Σ sim(u,v) * rating(v,p)  /  Σ sim(u,v)
#     → neighbours with higher similarity contribute more to the prediction.
#  3. Rank all unseen products by predicted score and return the top N.
def get_recommendations(target_user, matrix, sim_df,
                        top_n_users=5, top_n_products=10):
    """
    Return a ranked list of top_n_products product IDs for target_user.
    Uses weighted average of similar users' ratings.
    Products already rated by target_user are excluded.
    """
    # Step 1 — get top-k neighbours sorted by similarity (self excluded)
    neighbours = (
        sim_df[target_user]
        .drop(target_user)              # remove self (similarity = 1, not useful)
        .sort_values(ascending=False)   # highest similarity first
        .head(top_n_users)              # keep only the top k
    )

    # Products the target user has already interacted with — never recommend these
    already_rated = set(
        matrix.columns[matrix.loc[target_user] > 0]
    )

    # Accumulators for the weighted score formula
    score_num = defaultdict(float)   # numerator:   Σ sim(u,v) * rating(v,p)
    score_den = defaultdict(float)   # denominator: Σ sim(u,v)  (for normalisation)

    for neighbour, sim_weight in neighbours.items():
        if sim_weight <= 0:          # skip neighbours with no overlap
            continue
        for product in matrix.columns:
            if product in already_rated:  # skip already-seen products
                continue
            rating = matrix.loc[neighbour, product]
            if rating > 0:            # only include products the neighbour actually rated
                score_num[product] += sim_weight * rating
                score_den[product] += sim_weight

    # Step 3 — normalise to get final predicted scores, then rank
    predicted = {
        p: score_num[p] / score_den[p]
        for p in score_num if score_den[p] > 0
    }

    # Sort products by predicted score (highest first) and return top N IDs
    ranked = sorted(predicted.items(), key=lambda x: x[1], reverse=True)
    return [p for p, _ in ranked[:top_n_products]]


# Quick demo — generate top-10 recommendations for user U000
sample_recs = get_recommendations('U000', user_item_matrix_filled, similarity_df,
                                  top_n_users=5, top_n_products=10)
print("Top-10 recommendations for U000:", sample_recs)


## 2.4  Train / Test Split

For every user we randomly hold out **20 %** of their rated products as the test set.  
The model is trained on the remaining 80 % and evaluated against the hidden items.

In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────────
# Strategy: for each user, randomly hide 20% of their rated products.
#   - The model trains on the remaining 80% (train matrix).
#   - We check whether the hidden 20% appear in the model's recommendations (test_data).
# This simulates 'how well would this system have served a user in the past?'
def train_test_split_users(matrix, test_ratio=0.2, random_state=42):
    """
    Hold out test_ratio of each user's rated products.
    Returns a zeroed-out train matrix and a dict {user: [held-out products]}.
    """
    rng = np.random.default_rng(random_state)  # seeded RNG for reproducibility
    train = matrix.copy()   # start with a full copy — we'll zero out test items
    test_data = {}

    for user in matrix.index:
        # Identify all products this user has rated (non-zero entries)
        rated = matrix.columns[matrix.loc[user] > 0].tolist()

        # Hold out at least 1 product even for users with very few ratings
        n_test = max(1, int(len(rated) * test_ratio))

        # Randomly select the held-out products
        held_out = rng.choice(rated, size=n_test, replace=False).tolist()
        test_data[user] = held_out

        # Zero-out held-out ratings in the train matrix (hide them from the model)
        train.loc[user, held_out] = 0.0

    return train, test_data


train_matrix, test_data = train_test_split_users(user_item_matrix_filled)

total_test = sum(len(v) for v in test_data.values())
print(f"Total held-out interactions : {total_test}")
print(f"Avg held-out per user       : {total_test/len(test_data):.1f}")


## 2.5  Evaluation Metrics

| Metric | Formula | Interpretation |
|---|---|---|
| **Precision@K** | (hits in top-K) / K | How many of the K recommendations are relevant |
| **Recall@K** | (hits in top-K) / (total relevant) | How many relevant items were found |
| **MAP** | mean of AP across users | Rewards correct recommendations ranked higher |

**Relevant** = products in the user's held out test set.

In [ ]:
# ── Evaluation Metric Functions ───────────────────────────────────────────

def precision_at_k(recommended, relevant, k):
    # Precision@K = how many of our top-K picks are actually relevant?
    # 'relevant' = the user's hidden test products (ground truth)
    # A hit occurs when a recommended product is in the relevant set.
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / k   # divide by K (not by hits) — penalises irrelevant picks


def recall_at_k(recommended, relevant, k):
    # Recall@K = of all relevant items, how many did we surface in the top-K?
    # Recall increases as K grows (wider net catches more relevant items).
    if not relevant:
        return 0.0
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / len(relevant)   # divide by total relevant (not by K)


def average_precision(recommended, relevant, k):
    """AP for one user: rewards hits that appear early in the ranked list.
    A hit at rank 1 contributes more than a hit at rank 10."""
    if not relevant:
        return 0.0
    hits, cumsum = 0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in set(relevant):
            hits += 1
            # Add precision at the current rank position (only counted on a hit)
            cumsum += hits / (i + 1)
    # Normalise by the number of relevant items (capped at K)
    return cumsum / min(len(relevant), k)


def evaluate(train_mat, test_dict, k=10, top_n_users=5):
    """
    Full evaluation pipeline:
      1. Recompute user similarity using ONLY training data (prevents data leakage).
      2. Generate top-K recommendations for every user.
      3. Compare against held-out test items and average the metrics across all users.
    """
    # Recompute similarity on training data only — the test ratings must stay hidden
    train_sim = cosine_similarity(train_mat)
    train_sim_df = pd.DataFrame(
        train_sim,
        index=train_mat.index,
        columns=train_mat.index
    )

    prec_list, rec_list, ap_list = [], [], []

    for user in train_mat.index:
        relevant = test_dict.get(user, [])   # ground-truth held-out products
        if not relevant:
            continue
        # Generate recommendations using the training similarity matrix
        recs = get_recommendations(user, train_mat, train_sim_df,
                                   top_n_users=top_n_users,
                                   top_n_products=k)
        prec_list.append(precision_at_k(recs, relevant, k))
        rec_list.append(recall_at_k(recs, relevant, k))
        ap_list.append(average_precision(recs, relevant, k))

    # Average each metric across all users
    return {
        f'Precision@{k}': round(np.mean(prec_list), 4),
        f'Recall@{k}'   : round(np.mean(rec_list),  4),
        'MAP'           : round(np.mean(ap_list),    4),
    }


# Evaluate at two cutoffs: K=5 (short list) and K=10 (longer list)
results_k10 = evaluate(train_matrix, test_data, k=10, top_n_users=5)
results_k5  = evaluate(train_matrix, test_data, k=5,  top_n_users=5)

print("=== Evaluation Results ===")
print(f"K = 5  : {results_k5}")
print(f"K = 10 : {results_k10}")


## 2.6  Coverage & Intra-List Diversity

- **Catalog Coverage**: fraction of all products that appear in at least one recommendation list.
- **Intra-List Diversity (ILD)**: average pairwise dissimilarity (1 − item cosine similarity) within each user's recommendation list, then averaged across users. Higher = more varied recommendations.

In [ ]:
# ── Catalog Coverage ─────────────────────────────────────────────────────
def catalog_coverage(matrix, sim_df, k=10, top_n_users=5):
    """Fraction of all products that appear in at least one recommendation list.
    Low coverage means the model only recommends a small subset of popular items
    (popularity bias), ignoring the long tail of the catalogue."""
    all_products = set(matrix.columns)   # full product catalogue
    recommended  = set()                 # union of all recommendation lists
    for user in matrix.index:
        recs = get_recommendations(user, matrix, sim_df,
                                   top_n_users=top_n_users,
                                   top_n_products=k)
        recommended.update(recs)         # add this user's recs to the global set
    return round(len(recommended) / len(all_products), 4)


# ── Intra-List Diversity ──────────────────────────────────────────────────
def intra_list_diversity(matrix, sim_df, k=10, top_n_users=5):
    """Average pairwise item dissimilarity within each user's recommendation list.
    Computed as 1 - item cosine similarity.
    0 = all recommended items are identical; 1 = maximally diverse list."""
    # Transpose the matrix: now rows = products, columns = users
    # This lets us compute product-to-product similarity
    item_sim = cosine_similarity(matrix.T)   # shape: (n_products, n_products)
    item_sim_df = pd.DataFrame(item_sim,
                               index=matrix.columns,
                               columns=matrix.columns)
    diversities = []
    for user in matrix.index:
        recs = get_recommendations(user, matrix, sim_df,
                                   top_n_users=top_n_users,
                                   top_n_products=k)
        if len(recs) < 2:   # need at least 2 items to form a pair
            continue
        # Compute similarity for every unique pair in the list
        pairs = [
            item_sim_df.loc[recs[i], recs[j]]
            for i in range(len(recs))
            for j in range(i + 1, len(recs))
        ]
        # Diversity = 1 - average similarity (higher = more varied)
        diversities.append(1 - np.mean(pairs))
    return round(np.mean(diversities), 4)


coverage  = catalog_coverage(user_item_matrix_filled, similarity_df, k=10)
diversity = intra_list_diversity(user_item_matrix_filled, similarity_df, k=10)

print(f"Catalog Coverage     : {coverage*100:.1f}% of products recommended")
print(f"Intra-List Diversity : {diversity:.4f}  (0=identical items, 1=maximally diverse)")


## 2.7 — Top-5 Recommendations per User (Sample)

We generate top-5 product recommendations for every user and enrich them with product names and categories.

In [ ]:
# ── Generate Top-5 Recommendations for Every User ────────────────────────
# Build a lookup table: ProductID → (ProductName, Category)
# Used to enrich the output with human-readable product information
product_lookup = product_data.set_index('ProductID')[['ProductName', 'Category']]

all_recs = []
for user in user_item_matrix_filled.index:
    # Get the top-5 product IDs recommended for this user
    recs = get_recommendations(user, user_item_matrix_filled, similarity_df,
                               top_n_users=5, top_n_products=5)
    for rank, pid in enumerate(recs, start=1):   # rank starts at 1 (not 0)
        row = {'UserID': user, 'Rank': rank, 'ProductID': pid}
        # Attach product name and category if the product exists in the lookup
        row.update(product_lookup.loc[pid].to_dict() if pid in product_lookup.index else {})
        all_recs.append(row)

recs_df = pd.DataFrame(all_recs)
recs_df.to_csv('recommendations_top5.csv', index=False)   # save for reporting

print("Recommendations saved to recommendations_top5.csv")
print(f"Total rows: {len(recs_df)}")
# Preview the first 3 users' recommendations
recs_df[recs_df['UserID'].isin(['U000', 'U001', 'U002'])]


## 2.8  Visualisations

### (a) User Similarity Heatmap

In [ ]:
# ── (a) User–User Similarity Heatmap ─────────────────────────────────────
# Darker red = higher cosine similarity between two users.
# The diagonal is masked out (self-similarity is always 1 and would dominate the scale).
plt.figure(figsize=(14, 11))
mask = np.eye(len(similarity_df), dtype=bool)   # True on the diagonal → hidden
sns.heatmap(
    similarity_df,
    mask=mask,
    cmap='YlOrRd',         # yellow (low) → red (high)
    vmin=0, vmax=1,
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Cosine Similarity'},
    xticklabels=True,
    yticklabels=True
)
plt.title('User–User Cosine Similarity Heatmap', fontsize=14, pad=12)
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7, rotation=0)
plt.tight_layout()
plt.savefig('part2_similarity_heatmap.png', dpi=150)
plt.show()


### (b) Evaluation Metrics Bar Chart (K = 5 vs K = 10)

In [ ]:
# ── (b) Evaluation Metrics Bar Chart — K=5 vs K=10 ──────────────────────
# Comparing two cutoffs illustrates the precision/recall trade-off:
#   K=5  → shorter list, higher precision, lower recall
#   K=10 → longer list, lower precision, higher recall
metrics_labels = ['Precision', 'Recall', 'MAP']
vals_k5  = [results_k5['Precision@5'],   results_k5['Recall@5'],   results_k5['MAP']]
vals_k10 = [results_k10['Precision@10'], results_k10['Recall@10'], results_k10['MAP']]

x = np.arange(len(metrics_labels))
width = 0.32   # width of each bar

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, vals_k5,  width, label='K = 5',  color='steelblue', edgecolor='white')
bars2 = ax.bar(x + width/2, vals_k10, width, label='K = 10', color='coral',      edgecolor='white')

ax.set_ylabel('Score')
ax.set_title('Collaborative Filtering — Evaluation Metrics')
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, fontsize=12)
ax.set_ylim(0, max(vals_k5 + vals_k10) * 1.3)   # extra headroom for annotations
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

# Annotate each bar with its exact numeric value
for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('part2_evaluation_metrics.png', dpi=150)
plt.show()


### (c) Coverage & Diversity Summary

In [ ]:
# ── (c) Full Evaluation Summary Table ────────────────────────────────────
# Collect all metrics (accuracy + beyond-accuracy) into one overview table.
summary = pd.DataFrame({
    'Metric': ['Precision@5', 'Recall@5', 'MAP (K=5)',
               'Precision@10', 'Recall@10', 'MAP (K=10)',
               'Catalog Coverage', 'Intra-List Diversity'],
    'Value': [
        results_k5['Precision@5'],
        results_k5['Recall@5'],
        results_k5['MAP'],
        results_k10['Precision@10'],
        results_k10['Recall@10'],
        results_k10['MAP'],
        coverage,
        diversity
    ]
})

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis('off')   # hide the axes — we only want the table
tbl = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.4, 1.6)   # widen and taller cells for readability
plt.title('Part 2 — Full Evaluation Summary', fontsize=13, pad=16)
plt.tight_layout()
plt.savefig('part2_summary_table.png', dpi=150)
plt.show()

print(summary.to_string(index=False))




### Key Observations
- The **85.5% sparsity** limits precision — users share few common products to anchor similarity estimates.
- **Recall@10 > Recall@5** as expected: a wider list captures more held-out items.
- **MAP** is low in sparse settings because correct items rarely appear near rank 1.
- **Coverage** shows what fraction of the catalogue can be surfaced by the system.
- These limitations are worth discussing in the report (Conceptual Question 1).